In [0]:
-- ---------- Dimensions ----------
CREATE OR REPLACE MATERIALIZED VIEW dim_league_clusters AS
SELECT
  lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  any_value(name) AS cluster_name,
  min(season) AS season_start,
  max(season) AS season_end
FROM workspace.sleeper_raw.sleeper_league_info_snapshot
GROUP BY lower(regexp_replace(coalesce(name, 'unknown'), '[^a-zA-Z0-9]+', '_'));

CREATE OR REPLACE MATERIALIZED VIEW dim_manager_roster_map AS
WITH ro AS (
  SELECT league_id, roster_id, owner_id
  FROM workspace.sleeper_raw.sleeper_rosters_snapshot
),
us AS (
  SELECT league_id AS u_league_id, user_id AS manager_user_id, display_name
  FROM workspace.sleeper_raw.sleeper_users_snapshot
),
li AS (
  SELECT league_id, season, name FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT
  ro.league_id,
  li.season,
  ro.roster_id,
  us.manager_user_id,
  coalesce(us.display_name, 'Unknown') AS manager_display_name,
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name
FROM ro
LEFT JOIN us ON ro.league_id = us.u_league_id AND ro.owner_id = us.manager_user_id
LEFT JOIN li ON ro.league_id = li.league_id;

CREATE OR REPLACE MATERIALIZED VIEW dim_players AS
WITH all_players AS (
  SELECT explode(transform(map_keys(players_points), k -> k)) AS player_id
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot
  UNION ALL
  SELECT player_id FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot
)
SELECT DISTINCT player_id,
       CAST(NULL AS STRING) AS full_name,
       CAST(NULL AS STRING) AS position,
       CAST(NULL AS STRING) AS team,
       CAST(NULL AS STRING) AS status,
       current_timestamp() AS updated_at
FROM all_players;

-- Optional placeholder reference curve (empty by default)
CREATE OR REPLACE MATERIALIZED VIEW dim_draft_pick_value AS
SELECT * FROM (
  SELECT CAST(NULL AS STRING) AS season,
         CAST(NULL AS INT) AS round,
         CAST(NULL AS INT) AS pick,
         CAST(NULL AS DOUBLE) AS value
) WHERE 1=0;

-- ---------- Facts ----------
CREATE OR REPLACE MATERIALIZED VIEW fact_team_week AS
WITH base AS (
  SELECT m.league_id, m.week, m.matchup_id, m.roster_id, m.points AS points_for
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
),
opp AS (
  SELECT a.league_id, a.week, a.matchup_id, a.roster_id,
         a.points_for, b.points_for AS points_against
  FROM base a
  LEFT JOIN base b
    ON a.league_id=b.league_id AND a.week=b.week
   AND a.matchup_id=b.matchup_id AND a.roster_id<>b.roster_id
),
li AS (
  SELECT league_id, season FROM workspace.sleeper_raw.sleeper_league_info_snapshot
),
wmed AS (
  SELECT o.league_id, l.season, o.week,
         percentile_approx(o.points_for, 0.5) AS med
  FROM opp o JOIN li l USING (league_id)
  GROUP BY o.league_id, l.season, o.week
)
SELECT o.league_id, l.season, o.week, o.roster_id,
       o.points_for, o.points_against,
       (o.points_for >= w.med) AS median_beat_flag
FROM opp o
JOIN li l ON o.league_id = l.league_id
JOIN wmed w ON o.league_id=w.league_id AND l.season=w.season AND o.week=w.week;

CREATE OR REPLACE MATERIALIZED VIEW fact_player_week AS
WITH src AS (
  SELECT m.league_id, li.season, m.week, m.roster_id,
         map_entries(m.players_points) AS entries
  FROM workspace.sleeper_raw.sleeper_matchups_snapshot m
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
rows AS (
  SELECT league_id, season, week, roster_id,
         transform(entries, e -> named_struct('player_id', e.key, 'points', e.value)) AS rows
  FROM src
),
exploded AS (
  SELECT league_id, season, week, roster_id, explode(rows) AS r
  FROM rows
)
SELECT league_id, season, week, roster_id,
       r.player_id, r.points,
       CAST(NULL AS DOUBLE) AS projected_points,
       CAST(NULL AS STRING) AS position,
       current_timestamp() AS updated_at
FROM exploded;

-- Enriched with cluster via league name (deterministic)
CREATE OR REPLACE MATERIALIZED VIEW fact_player_week_enriched AS
SELECT f.*,
       lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
       li.name AS cluster_name
FROM fact_player_week f
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

CREATE OR REPLACE MATERIALIZED VIEW fact_standings_week AS
WITH wl AS (
  SELECT league_id, season, week, roster_id,
         (points_for > points_against)  AS win_flag,
         (points_for < points_against)  AS loss_flag,
         (points_for = points_against)  AS tie_flag,
         median_beat_flag,
         points_for, points_against
  FROM fact_team_week
)
SELECT league_id, season, week, roster_id,
       SUM(CASE WHEN win_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS wins,
       SUM(CASE WHEN loss_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS losses,
       SUM(CASE WHEN tie_flag  THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ties,
       points_for, points_against,
       SUM(CASE WHEN median_beat_flag THEN 1 ELSE 0 END)
         OVER (PARTITION BY league_id, season, roster_id ORDER BY week
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS expected_wins
FROM wl;

-- ---------- Events ----------
CREATE OR REPLACE MATERIALIZED VIEW evt_waiver_events AS
WITH tx AS (
  SELECT league_id, transaction_id, type, status, adds, drops, waiver_bid, leg, created
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot
),
adds AS (
  SELECT league_id, transaction_id, waiver_bid, leg, created,
         explode(transform(map_entries(adds),
                 e -> named_struct('player_id', e.key, 'to_roster_id', e.value))) AS a
  FROM tx
),
drops AS (
  SELECT league_id, transaction_id, waiver_bid, leg, created,
         explode(transform(map_entries(drops),
                 e -> named_struct('player_id', e.key, 'from_roster_id', e.value))) AS d
  FROM tx
)
SELECT league_id, transaction_id,
       a.player_id, a.to_roster_id, CAST(NULL AS INT) AS from_roster_id,
       waiver_bid AS faab_delta,
       to_timestamp(created/1000.0) AS event_ts,
       leg AS week,
       'add' AS action
FROM adds
UNION ALL
SELECT league_id, transaction_id,
       d.player_id, CAST(NULL AS INT) AS to_roster_id, d.from_roster_id,
       waiver_bid AS faab_delta,
       to_timestamp(created/1000.0) AS event_ts,
       leg AS week,
       'drop' AS action
FROM drops;

-- ---------- Aggregates ----------
CREATE OR REPLACE MATERIALIZED VIEW agg_consistency_metrics AS
SELECT league_id, season, roster_id,
       stddev_pop(points_for) AS points_for_stddev,
       avg(points_for)       AS points_for_avg,
       count(*)              AS games_played,
       CASE WHEN avg(points_for) <> 0 THEN stddev_pop(points_for)/avg(points_for) END AS points_for_cv
FROM fact_team_week
GROUP BY league_id, season, roster_id;

CREATE OR REPLACE MATERIALIZED VIEW agg_rivalry_head_to_head AS
WITH a AS (SELECT * FROM fact_team_week),
     b AS (SELECT * FROM fact_team_week)
SELECT a.league_id, a.season,
       a.roster_id AS roster_id_a, b.roster_id AS roster_id_b,
       count(*) AS games_played,
       sum(CASE WHEN a.points_for > b.points_for THEN 1 ELSE 0 END) AS wins_a,
       sum(CASE WHEN a.points_for < b.points_for THEN 1 ELSE 0 END) AS wins_b,
       sum(CASE WHEN a.points_for = b.points_for THEN 1 ELSE 0 END) AS ties,
       avg(a.points_for) AS pf_per_game_a,
       avg(b.points_for) AS pf_per_game_b
FROM a JOIN b
  ON a.league_id=b.league_id AND a.season=b.season AND a.week=b.week
 AND a.roster_id<>b.roster_id
GROUP BY a.league_id, a.season, a.roster_id, b.roster_id;

CREATE OR REPLACE MATERIALIZED VIEW agg_records_all_time AS
SELECT league_id, roster_id,
       max(points_for) AS max_points_for,
       min(points_for) AS min_points_for
FROM fact_team_week
GROUP BY league_id, roster_id;

-- Waiver steals (4-week window)
CREATE OR REPLACE MATERIALIZED VIEW agg_waiver_steals_window AS
WITH tx AS (
  SELECT * FROM workspace.sleeper_raw.sleeper_transactions_snapshot WHERE type IN ('waiver','free_agent')
),
adds AS (
  SELECT league_id, leg AS week, explode(transform(map_entries(adds), e -> named_struct('player_id', e.key, 'roster_id', e.value))) AS a, created
  FROM tx
),
drops AS (
  SELECT league_id, leg AS week, explode(transform(map_entries(drops), e -> named_struct('player_id', e.key, 'roster_id', e.value))) AS d, created
  FROM tx
),
li AS (SELECT league_id, season FROM workspace.sleeper_raw.sleeper_league_info_snapshot),
fpw AS (SELECT * FROM fact_player_week),
add_realized AS (
  SELECT a.league_id, l.season, a.week, a.a.roster_id AS roster_id, a.a.player_id AS player_id,
         SUM(p.points) AS added_points
  FROM adds a JOIN li l USING(league_id)
  JOIN fpw p ON a.league_id=p.league_id AND l.season=p.season AND a.a.roster_id=p.roster_id
            AND p.week > a.week AND p.week <= a.week+4
  GROUP BY a.league_id, l.season, a.week, a.a.roster_id, a.a.player_id
),
drop_realized AS (
  SELECT d.league_id, l.season, d.week, d.d.roster_id AS roster_id, d.d.player_id AS player_id,
         SUM(p.points) AS dropped_points
  FROM drops d JOIN lI l USING(league_id)
  JOIN fpw p ON d.league_id=p.league_id AND l.season=p.season AND d.d.roster_id=p.roster_id
            AND p.week > d.week AND p.week <= d.week+4
  GROUP BY d.league_id, l.season, d.week, d.d.roster_id, d.d.player_id
)
SELECT a.league_id, a.season, a.week AS window_start_week, a.week+4 AS window_end_week,
       a.roster_id, a.player_id, a.added_points, dr.dropped_points,
       coalesce(a.added_points,0) - coalesce(dr.dropped_points,0) AS realized_points_delta
FROM add_realized a
LEFT JOIN drop_realized dr
  ON a.league_id=dr.league_id AND a.season=dr.season AND a.week=dr.week AND a.roster_id=dr.roster_id;

-- Per-LEAGUE, per-SEASON, per-ROUND ROI
CREATE OR REFRESH MATERIALIZED VIEW agg_draft_roi_by_round AS
WITH picks AS (
  SELECT
    p.league_id,
    li.season,
    CAST(p.round AS INT) AS round,
    p.player_id
  FROM workspace.sleeper_raw.sleeper_draft_picks_snapshot p
  JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id)
),
player_season_points AS (
  SELECT
    league_id, season, player_id,
    SUM(points) AS season_points
  FROM fact_player_week
  GROUP BY league_id, season, player_id
),
rook AS (
  SELECT
    pk.league_id, pk.season, pk.round, pk.player_id,
    COALESCE(psp.season_points, 0.0) AS season_points
  FROM picks pk
  LEFT JOIN player_season_points psp
    ON pk.league_id = psp.league_id
   AND pk.season    = psp.season
   AND pk.player_id = psp.player_id
),
by_round AS (
  SELECT
    league_id, season, round,
    AVG(season_points) AS avg_points
  FROM rook
  GROUP BY league_id, season, round
),
season_avg AS (
  SELECT
    league_id, season,
    AVG(season_points) AS season_avg_points
  FROM rook
  GROUP BY league_id, season
)
SELECT
  b.league_id,
  b.season,
  b.round,
  b.avg_points,
  s.season_avg_points,
  CASE WHEN s.season_avg_points > 0
       THEN b.avg_points / s.season_avg_points
       ELSE NULL
  END AS avg_roi
FROM by_round b
JOIN season_avg s
  ON b.league_id = s.league_id AND b.season = s.season;
